In [6]:
%pip install -q numpy pandas


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Config

In [7]:
from src.constants import Column, INPUT_DIR, OUTPUT_DIR

# Load dataset

In [8]:
import pandas as pd

electronics_df = pd.read_csv(f"{INPUT_DIR}/electronics_raw.csv")

result_df = pd.DataFrame()

# Reusable functions

In [9]:
import re 
import numpy as np

# AI generated Regex pattern to seperate words and sentences
WORD_PATTERN = re.compile(r"\b[A-Za-z0-9]+(?:['’-][A-Za-z0-9]+)*\b") 
SENTENCE_PATTERN = re.compile(r"[^.!?]+(?:[.!?]+|$)")

FIRST_PERSON_PRONOUNS = {
    "i", "me", "my", "mine", "myself",
    "we", "us", "our", "ours", "ourselves",
}
SECOND_PERSON_PRONOUNS = {
    "you", "your", "yours", "yourself", "yourselves",
}
THIRD_PERSON_PRONOUNS = {
    "he", "him", "his", "himself",
    "she", "her", "hers", "herself",
    "it", "its", "itself",
    "they", "them", "their", "theirs", "themselves",
}
CONTRACTION_PRONOUNS = {
    "i", "we", "you", "he", "she", "it", "they",
}

def calculate_word_count(text: str) -> int:
    return len(WORD_PATTERN.findall(str(text)))

def calculate_sentence_count(text: str) -> int:
    sentences = [
        match.group().strip()
        for match in SENTENCE_PATTERN.finditer(str(text))
        if match.group().strip()
    ]

    return len(sentences)

def calculate_character_count(text: str) -> int:
    return sum(character.isalnum() for character in str(text))

def calculate_readability_ari(text: str) -> float:
    word_count = calculate_word_count(text)
    sentence_count = calculate_sentence_count(text)
    character_count = calculate_character_count(text)

    if word_count == 0 or sentence_count == 0:
        return np.nan

    return (
        4.71 * character_count / word_count
        + 0.5 * word_count / sentence_count
        - 21.43
    )

def calculate_extremity(rating: float) -> int:
    return int(rating in {1, 5})

def calculate_personal_pronoun_counts(text: str) -> tuple[int, int, int]:
    first_person_count = 0
    second_person_count = 0
    third_person_count = 0

    for original_token in WORD_PATTERN.findall(str(text)):
        # Avoid counting common abbreviations as the pronouns 'us' or 'it'.
        if original_token in {"US", "IT"}:
            continue

        token = original_token.lower().replace("’", "'")
        contraction_base = token.split("'", maxsplit=1)[0]
        if contraction_base in CONTRACTION_PRONOUNS:
            token = contraction_base

        first_person_count += token in FIRST_PERSON_PRONOUNS
        second_person_count += token in SECOND_PERSON_PRONOUNS
        third_person_count += token in THIRD_PERSON_PRONOUNS

    return first_person_count, second_person_count, third_person_count

# Apply Deterministic measured Columns

In [10]:
result_df[Column.ID] = electronics_df[Column.ID]
result_df[Column.REVIEW_LENGTH] = electronics_df[Column.TEXT].apply(calculate_word_count)
result_df[Column.SENTENCE_COUNT] = electronics_df[Column.TEXT].apply(calculate_sentence_count)
result_df[Column.CHARACTER_COUNT] = electronics_df[Column.TEXT].apply(calculate_character_count)
result_df[Column.READABILITY_ARI] = electronics_df[Column.TEXT].apply(calculate_readability_ari)

result_df[Column.EXTREMITY] = electronics_df[Column.RATING].apply(calculate_extremity)

pronoun_counts = electronics_df[Column.TEXT].apply(calculate_personal_pronoun_counts)
result_df[[
    Column.FIRST_PERSON_PRONOUN_COUNT,
    Column.SECOND_PERSON_PRONOUN_COUNT,
    Column.THIRD_PERSON_PRONOUN_COUNT,
]] = pd.DataFrame(pronoun_counts.tolist(), index=result_df.index)

# Write to CSV

In [11]:
result_df.to_csv(
    f"{OUTPUT_DIR}/deterministic_features_results.csv", 
    index=False, 
    encoding="utf-8"
)